In [1]:
from IPython.display import HTML

# standard imports
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
import math
from math import pi
np.set_printoptions(
    linewidth=120, formatter={
        'float': lambda x: f"{0:8.4g}" if abs(x) < 1e-10 else f"{x:8.4g}"})
np.random.seed(0)
from spatialmath import *
from spatialmath.base import *

### Transforming Spatial Velocities

In [2]:
aTb = SE3.Tx(-2) * SE3.Rz(-pi/2) * SE3.Rx(pi/2)
aTb

   0         0        -1        -2         
  -1         0         0         0         
   0         1         0         0         
   0         0         0         1         


In [3]:
bV = [1, 2, 3, 4, 5, 6]

In [4]:
aJb = aTb.jacob()
aJb.shape

(6, 6)

In [5]:
aJb

array([[       0,        0,       -1,        0,        0,        0],
       [      -1,        0,        0,        0,        0,        0],
       [       0,        1,        0,        0,        0,        0],
       [       0,        0,        0,        0,        0,       -1],
       [       0,        0,        0,       -1,        0,        0],
       [       0,        0,        0,        0,        1,        0]])

In [6]:
aV = aJb @ bV
aV

array([      -3,       -1,        2,       -6,       -4,        5])

In [7]:
aV = aTb.Ad() @ [1, 2, 3, 0, 0, 0]
aV

array([      -3,       -1,        2,        0,        0,        0])

In [8]:
aV = aTb.Ad() @ [0, 0, 0, 1, 0, 0]
aV

array([       0,        0,        2,        0,       -1,        0])

In [9]:
aV = aTb.Ad() @ [1, 2, 3, 1, 0, 0]
aV

array([      -3,       -1,        4,        0,       -1,        0])

### Incremental Rotation

In [10]:
rotx(0.001)

array([[       1,        0,        0],
       [       0,        1,   -0.001],
       [       0,    0.001,        1]])

In [11]:
import time

Rexact = np.eye(3) # null rotation
Rapprox = np.eye(3) # null rotation
w = np.array([1, 0, 0]) # rotation of 1 rad/s about x-axis
dt = 0.01 # time step
t0 = time.process_time()
for i in range(100): # exact integration over 100 time steps
    Rexact = Rexact @ trexp(skew(w*dt)) # update by composition
print(time.process_time() - t0)

0.046875


In [12]:
t0 = time.process_time()
for i in range(100): # approx. integration over 100 steps
    Rapprox += Rapprox @ skew(w*dt)
print(time.process_time() - t0)

0.0


In [13]:
Rexact

array([[       1,        0,        0],
       [       0,   0.5403,  -0.8415],
       [       0,   0.8415,   0.5403]])

In [14]:
Rapprox

array([[       1,        0,        0],
       [       0,    0.543,  -0.8457],
       [       0,   0.8457,    0.543]])

In [15]:
np.linalg.det(Rapprox) - 1

0.010049662092876055

In [16]:
np.linalg.det(Rexact) - 1

-2.886579864025407e-15

In [17]:
tr2angvec(trnorm(Rexact))

(1.0, array([       1,        0,        0]))

In [18]:
tr2angvec(trnorm(Rapprox))

(0.999966668666524, array([       1,        0,        0]))